In [2]:
!pip install -q -U torchao
!pip install -q pytrec_eval
!pip install -q rank-bm25

!pip install -q colpali-engine
!pip install -q open_clip_torch==2.23.0
!pip install -q git+https://github.com/Mahmoodlab/CONCH.git


  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.3/112.3 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 29.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [3]:
import shutil
import os

def updated_repo():
    repo_path = "/content/retrieval-benchmark"

    if os.path.exists(repo_path):
        shutil.rmtree(repo_path)

    get_ipython().system(
        "git clone https://github.com/Archit381/retrieval-benchmark.git"
    )

In [4]:
import sys

updated_repo()

sys.path.append("/content/retrieval-benchmark")


Cloning into 'retrieval-benchmark'...
remote: Enumerating objects: 96, done.
remote: Counting objects: 100% (96/96), done.
remote: Compressing objects: 100% (63/63), done.
remote: Total 96 (delta 46), reused 80 (delta 30), pack-reused 0 (from 0)
Receiving objects: 100% (96/96), 21.92 KiB | 10.96 MiB/s, done.
Resolving deltas: 100% (46/46), done.


In [5]:
import io
import os
import json

import torch
from PIL import Image as PILImage
import pyarrow as pa
import pyarrow.parquet as pq
from datasets import load_dataset
from huggingface_hub import HfApi

from src.embedding_factory import EmbeddingFactory
from src.core import (
    SetInfo,
    Manifest,
    _decode_hf_image,
    print_embedding_output,
    save_artifacts
)

SOURCE_DATASET = "architojha/m3-retrieve-csr-cardiology-1k"
HF_REPO_ID     = "architojha/m3-retrieve-eval"
ARTIFACTS_DIR  = "artifacts"
DEVICE         = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Device: {DEVICE}")

dataset = load_dataset(SOURCE_DATASET)

/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


Device: cuda


README.md:   0%|          | 0.00/508 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/224M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [6]:
def prepare_retrieval_data(df):
    # Build qrels
    qrels = {}
    for _, row in df.iterrows():
        qid = str(row["query_id"])
        did = str(row["pos_corpus_id"])
        qrels.setdefault(qid, {})[did] = int(row["score"])

    # Queries
    query_df = (
        df[["query_id", "image_path", "query_text"]]
        .drop_duplicates("query_id")
        .reset_index(drop=True)
    )

    query_ids = [str(x) for x in query_df["query_id"]]
    query_captions = query_df["query_text"].tolist()
    query_images = [
        _decode_hf_image(img)
        for img in query_df["image_path"].tolist()
    ]

    # Documents
    doc_df = (
        df[["pos_corpus_id", "pos_text"]]
        .drop_duplicates("pos_corpus_id")
        .reset_index(drop=True)
    )

    doc_ids = [str(x) for x in doc_df["pos_corpus_id"]]
    doc_texts = doc_df["pos_text"].tolist()

    # Validation
    assert len(query_ids) == len(query_images) == len(query_captions)
    assert len(doc_ids) == len(doc_texts)

    assert len(set(query_ids)) == len(query_ids), \
        "duplicate query_ids after dedup"
    assert len(set(doc_ids)) == len(doc_ids), \
        "duplicate doc_ids after dedup"

    query_id_set = set(query_ids)
    doc_id_set = set(doc_ids)

    for qid, rels in qrels.items():
        assert qid in query_id_set, \
            f"qrel qid {qid!r} missing from query list"

        for did in rels:
            assert did in doc_id_set, \
                f"qrel did {did!r} missing from doc list"

    print(f"unique queries : {len(query_ids)}")
    print(f"unique docs    : {len(doc_ids)}")
    print(f"qrel pairs     : {sum(len(v) for v in qrels.values())}")

    return {
        "qrels": qrels,
        "query_ids": query_ids,
        "query_captions": query_captions,
        "query_images": query_images,
        "doc_ids": doc_ids,
        "doc_texts": doc_texts,
    }



df = dataset["train"].to_pandas()
data = prepare_retrieval_data(df)

qrels = data["qrels"]
query_ids = data["query_ids"]
query_captions = data["query_captions"]
query_images = data["query_images"]
doc_ids = data["doc_ids"]
doc_texts = data["doc_texts"]

unique queries : 58
unique docs    : 561
qrel pairs     : 1000


In [16]:
model_map = {
    "colsmol": "vidore/colSmol-500M",
    "biomedclip": "microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224",
    "conch": "MahmoodLab/conch",
    "pubmedclip": "flaviagiammarino/pubmed-clip-vit-base-patch32"
}

In [ ]:
factory = EmbeddingFactory(device=DEVICE, dtype=torch.bfloat16)

local_embds_list = {}
for model_type, model_name in model_map.items():

    # if model_type != "pubmedclip":
    #   continue

    embd_out = factory.generate_sets(
        model_name,
        sets={
            "query_image":   {"images": query_images},
            "query_caption": {"texts":  query_captions},
            "doc":           {"texts":  doc_texts},
        },
        batch_size=4,
    )

    local_embds_list[model_type] = embd_out

    save_artifacts(
          model_name=model_name,
          model_type=model_type,
          sets_out=embd_out,
          roles={
              "query_image":   "query",
              "query_caption": "query",
              "doc":           "doc",
          },
          query_ids=query_ids,
          doc_ids=doc_ids,
          qrels=qrels,
          artifacts_dir=ARTIFACTS_DIR,
          hf_repo_id=HF_REPO_ID,
          df=df,
      )

In [ ]:
# print_embedding_output(embd_out['query_image'])
# print_embedding_output(embd_out['query_caption'])
# print_embedding_output(embd_out['doc'])

In [ ]:
api = HfApi(token='')

api.create_repo(HF_REPO_ID, repo_type="dataset", exist_ok=True)
api.upload_folder(
    folder_path=ARTIFACTS_DIR,
    repo_id=HF_REPO_ID,
    repo_type="dataset",
)
print(f"Artifacts pushed to hf://datasets/{HF_REPO_ID}")

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...query_caption.safetensors: 100%|##########|  119kB /  119kB            

  ...p/query_image.safetensors: 100%|##########|  119kB /  119kB            

  ...query_caption.safetensors: 100%|##########|  119kB /  119kB            

...

  ...iomedclip/doc.safetensors: 100%|##########| 1.15MB / 1.15MB            

  ...ubmedclip/doc.safetensors: 100%|##########| 1.15MB / 1.15MB            

  ...l/query_image.safetensors: 100%|##########| 33.6MB / 33.6MB            

No files have been modified since last commit. Skipping to prevent empty commit.


Artifacts pushed to hf://datasets/architojha/m3-retrieve-eval


In [54]:
from src.evaluation_factory import evaluate

eval_results = {}
for model_type in local_embds_list.keys():
    embd_out = local_embds_list[model_type]

    result = evaluate(
          model_type=model_type,
          query_embs=embd_out["query_image"].img_embd,
          doc_embs=embd_out["doc"].text_embd,
          query_ids=query_ids,
          doc_ids=doc_ids,
          qrels=qrels,
          device=DEVICE
      )

    eval_results[model_type] = result

In [62]:
ndcg_df = compare_ndcg(eval_results)

,nDCG@10
Model,
BiomedCLIP,0.265
PubMedCLIP,0.083 (-68.6\%)
ColSmol-500M,0.074 (-72.0\%)
CONCH,0.033 (-87.7\%)
